In [ ]:
import xhermes
from xhermes.selectors import selector_poloidal, selector_radial, selector_2d
from xhermes.plotting import plot_selection

%load_ext autoreload
%autoreload 2

# How to use this tutorial
To keep the GitHub history clean, this notebook has no outputs saved. Please replace the simulation path with your own and re-run the cells.
In the near future, we will add a test dataset on Zenodo which can be downloaded automatically for use with tutorials.

# Executive summary
This notebook introduces the basics of region selection in xHermes. Currently, only 2D regions and boundaries are supported, but selecting arbitrary radial and poloidal cell rings is so far only implemented in `sdtools` (https://github.com/mikekryjak/sdtools/blob/main/hermes3/selectors.py). This will be ported to xHermes in the near future.

### Tools covered:
`selector_poloidal` returns a poloidal index or slice, e.g.

>xhermes.selector_poloidal("sol")

`selector_radial` returns a radial index or slice, e.g.

>xhermes.selector_radial("domain")

`xhermes.selector_2d` returns a tuple of indices/slices and takes both a radial and poloidal selection. It has handling of incompatible combinations and can automatically parse a `boundary` radial selection into a radial boundary appropriate to the poloidal region. e.g.

>xhermes.selector_2d("boundary", "sol")

The selectors can be used to select a region of the dataset using the same syntax as `selector_2d`, e.g.
>ds.hermes.select_region("domain", "outer_lower_target")

This works for a `DataArray`, `Dataset` and `xhermes.HypnotoadGrid`.

This can be plotted to visually check what is being selected:

>xhermes.plot_selection("domain", "outer_lower_target")



# Read dataset and select the final time slice
Replace the paths with your own simulation

In [ ]:
ds = xhermes.open(
    r"/home/mike/work/cases/mastu2d/m9ax-10core_perftest",  ## Replace with your path to 1D-threshold solution
    gridfilepath="/home/mike/work/cases/mu1af6-tunepuff.nc",
    keep_yboundaries=True,
    keep_xboundaries=True,
    geometry="toroidal",
)

ds = (
    ds.bout.final_time()
)  # Update your xBOUT version to the latest master branch if this is missing!

# Poloidal selections
Use `selector_poloidal` to return a slice or integer array defining a poloidal location.

Example:

In [ ]:
selector_poloidal(ds, "outer_upper_sol")

Some notes on convention:
- Single and double null topologies have different regions. This is handled by double null having extra region options, e.g. `lower_pfr` and `upper_pfr`. Some region names are topology agnostic, e.g. `pfr` will select all PFR regions in the domain.
- "upper" and "lower" midplane refer to the sides of the cell edge exactly at the midplane.
- "extra", e.g. "inner_lower_sol_extra", refers to a single leg SOL slice which extends one cell beyond the midplane (so you can interpolate exactly onto the midplane if required)
- "guards" includes all guard cells. If "guards" is not present in the name, guards are never present.

A complete list of locations can be obtained with:

In [ ]:
selector_poloidal(ds, return_available=True)

# Radial selections
Use `selector_radial` to return a slice or integer array defining a radial location. 
The names `xlow` and `xup` refer to the direction towards the start and end of the radial index, so towards the core/pfr and SOL, respectively.

Example:

In [ ]:
selector_radial(ds, "boundary_xup")

The following radial regions are supported:
- domain (excluding guards)
- domain_guards
- boundary_xlow
- boundary_guard_xlow
- boundary_xup
- boundary_guard_xup


As with `selector_poloidal`, the available regions can be shown with:

In [ ]:
selector_radial(ds, return_available=True)

# 2D selections

`selector_2d` combines poloidal and radial selections with some extra features:
- Supports radial region `boundary` which automatically resolves to `xup`/`xlow`
- Raises exceptions for invalid combinations (e.g. selecting guards if none present, or xlow radial boundary for SOL poloidal region)
Example:

In [ ]:
selector_2d(ds, "boundary", "outer_upper_sol")

# Visualising selections
Use `plot_selection` to visualise the selection on both the logical and poloidal grid as selected by `select_2d`.
By default, it takes the name of the radial and poloidal regions. You can override this
and provide a custom selection slice/array instead.


Example: plotting the outer upper SOL boundary

In [ ]:
plot_selection(ds, "domain", "outer_sol")

Example: plotting a custom region

In [ ]:
plot_selection(ds, custom_selection=(slice(0, 20), slice(20, 24)))

# Using selections to get data

You can use the same selection syntax to slice the dataset using `ds.select_region(radial_region, poloidal_region)`. You can also do this on an individual `DataArray`, e.g. `ds["Te"].select_region(radial_region, poloidal_region)`.

### NOTE:
Note that `select_poloidal`, `select_radial` and `select_2d` return selection indices/slices, while `ds.select_region` returns the sliced dataset.

Example: plotting density on the outer lower target

In [ ]:
ds_outer_lower_target = ds.hermes.select_region("domain", "outer_lower_target")
ds_outer_lower_target["Ne"].bout.final_time().plot()

Example: 2D plot of density on a custom selection

In [ ]:
custom_region = ds.hermes.select_region(custom_selection=(slice(0, 20), slice(20, 24)))
custom_region["Te"].plot()